# Fine-tune the pretrained encoder on a Colab GPU

Runs Task 4 of the M5 plan (`docs/plans/2026-10-02-m5-finetuned-model.md`) on a free GPU
instead of the M1 Mac, with the same commands as `scripts/train_finetune.py` documents:

1. **Selection** (design 4.5): one epoch, capped at 20,000 windows, for `microsoft/deberta-v3-xsmall` and `distilbert-base-cased`.
2. **Full runs**, after decision 0009 names the encoder: `finetuned` (3 epochs) and `finetuned-ablation` (same model, random initialization).

**Before running:** Runtime > Change runtime type > T4 GPU. Then run the cells in order.

**Inputs:** the git branch (cell 2) and `data/clean/train.jsonl`, zipped as `clean-train.zip` (cell 3; make it locally with `cd data/clean && zip -j ../clean-train.zip train.jsonl`). Upload the zip to `My Drive/newline-fixer/` or upload it when cell 3 asks.

**Outputs:** `experiments/runs/<run-id>/` (weights, git-ignored) and `experiments/training/<run-id>.json` (the committed record). Both are copied to `My Drive/newline-fixer/` every two minutes and at the end of each run, and zipped for download.

In [ ]:
import subprocess, sys
assert sys.version_info >= (3, 12), f"the project needs Python 3.12+, Colab has {sys.version}"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "NO GPU: set the runtime type to T4 GPU")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
REPO = "https://github.com/jalalhussein1982/newline-fixer.git"
BRANCH = "m5-finetuned-model"  # the branch that holds the fine-tuning trainer

import os, shutil
if os.path.isdir("/content/newline-fixer"):
    shutil.rmtree("/content/newline-fixer")
!git clone --quiet --branch {BRANCH} {REPO} /content/newline-fixer
%cd /content/newline-fixer
# Colab's own torch carries CUDA; the project's uv.lock pins a CPU-only torch on Linux for CI,
# so install the package without resolving its extras.
!pip install --quiet --no-deps -e .
!pip install -q sentencepiece protobuf  # tokenizers for DeBERTa-v3; Colab ships torch and transformers
!git log --oneline -1

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/newline-fixer")
DRIVE.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE / "clean-train.zip"
if not zip_path.exists():
    print("clean-train.zip not found in Drive; choose it from your computer")
    uploaded = files.upload()
    name = next(iter(uploaded))
    shutil.move(name, zip_path)  # Drive is another filesystem: rename would fail

Path("data/clean").mkdir(parents=True, exist_ok=True)
!unzip -q -o "{zip_path}" -d data/clean
!ls -la data/clean data/sets

In [ ]:
import json, shutil, subprocess, threading, time
from pathlib import Path

RUNS, RECORDS = Path("experiments/runs"), Path("experiments/training")


def sync(run_id: str) -> None:
    """Copy the run directory and the record to Drive (idempotent)."""
    if (RUNS / run_id).exists():
        shutil.copytree(RUNS / run_id, DRIVE / "runs" / run_id, dirs_exist_ok=True)
    if (RECORDS / f"{run_id}.json").exists():
        (DRIVE / "training").mkdir(exist_ok=True)
        shutil.copy(RECORDS / f"{run_id}.json", DRIVE / "training" / f"{run_id}.json")


def train(
    run_id: str,
    model: str,
    epochs: int = 3,
    max_examples: int | None = None,
    random_init: bool = False,
    every: int = 120,
) -> None:
    """scripts/train_finetune.py on CUDA, with a Drive sync every `every` seconds and at the end."""
    cmd = [
        "python", "-u", "scripts/train_finetune.py", "--run-id", run_id, "--model", model,  # -u: print each epoch as it ends
        "--epochs", str(epochs), "--seed", "1", "--device", "cuda",
    ]
    if max_examples is not None:
        cmd += ["--max-examples", str(max_examples)]
    if random_init:
        cmd.append("--random-init")
    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    stop = threading.Event()

    def syncer() -> None:
        while not stop.wait(every):
            sync(run_id)

    threading.Thread(target=syncer, daemon=True).start()
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    stop.set()
    sync(run_id)
    if code != 0:
        raise RuntimeError(f"{run_id} exited with {code}")
    record = json.loads((RUNS / run_id / "run.json").read_text())
    print(f"\n{run_id}: best epoch {record['best_epoch']}, {record['seconds'] / 60:.1f} min, best {record['best']}")

## Selection runs (design 4.5)

One epoch each, 20,000 windows. Back on the Mac, `scripts/select_encoder.py` adds the CPU latency per 256-token window.

In [ ]:
train("ft-deberta-select", "microsoft/deberta-v3-xsmall", epochs=1, max_examples=20000)

In [ ]:
train("ft-distilbert-select", "distilbert-base-cased", epochs=1, max_examples=20000)

In [ ]:
# Bundle both selection runs and their records; unzip at the repo root on the Mac.
!rm -f m5-select.zip && zip -q -r m5-select.zip experiments/runs/ft-deberta-select experiments/runs/ft-distilbert-select experiments/training/ft-deberta-select.json experiments/training/ft-distilbert-select.json
shutil.copy("m5-select.zip", DRIVE / "m5-select.zip")
!ls -la m5-select.zip
files.download("m5-select.zip")

## Set MODEL after decision 0009

Stop here until decision 0009 has chosen the encoder from the selection numbers, then set `MODEL` below and run the two full runs.

In [ ]:
MODEL = "microsoft/deberta-v3-xsmall"  # replace with decision 0009's choice

In [ ]:
train("finetuned", MODEL, epochs=3)

In [ ]:
train("finetuned-ablation", MODEL, epochs=3, random_init=True)

In [ ]:
# Bundle both full runs and their records; unzip at the repo root on the Mac.
!rm -f m5-runs.zip && zip -q -r m5-runs.zip experiments/runs/finetuned experiments/runs/finetuned-ablation experiments/training/finetuned.json experiments/training/finetuned-ablation.json
shutil.copy("m5-runs.zip", DRIVE / "m5-runs.zip")
!ls -la m5-runs.zip
files.download("m5-runs.zip")

Back on the Mac, from the repo root:

```bash
unzip -o ~/Downloads/m5-select.zip   # after the selection runs
unzip -o ~/Downloads/m5-runs.zip     # after the full runs
```

Then `uv run python scripts/select_encoder.py --runs ft-deberta-select,ft-distilbert-select --out experiments/results/m5-candidates.json` for the selection table, and evaluate the full runs.